## Overview: data preparation for individual-level regressions

Note: the resulting tables (commits, users, projects) are rather big. Therefore, all the preliminary preparation, cleaning and merging is done on the Cloud. Only clean and short task-specific tables are used in Stata to run regressions and to prepare summary statistics. 

* Regression preparations 
    * Outcomes: productivity = number of commits, number of stars to projects owned, number of project forks
    * Level: annual
    * Sample Ukrainian (Russian) users; CEM: other UA/RU users without UA/RU collaboration before or CEM: control countries
    * Heterogeneity: productivity of contributors and projects --> quality of c and pm previous work on given project, identity, name_identity, experience with russian/ukrainian owner generally (extensive), share of previous collab with ru/ua (intensive) --> outside options, users joining before 2014 --> dummy
   

In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

In [ ]:
# outcomes
    # total commits 
    # commits weighted by project's cumulated quality
    # n_owners and n_countries (unique per year)
    # total own projects
    # projects weighted by stars 
    # n_contributors and n_contributing countries (unique per year) 
query = '''
CREATE TABLE `friends-foes.commits.ind_commits`
AS
# total commits and commits weighted by project's cum quality in t-1
SELECT year, author_id,
       SUM(commits) as commits_sent,
       SUM(IF(country_owner = 'ru' AND country_author != 'ru', commits, NULL)) as commits_toru, 
       SUM(IF(country_owner = 'ua' AND country_author != 'ua', commits, NULL)) as commits_toua, 
       SUM(IF(country_owner = country_author, commits, NULL)) as commits_tohome, 
       SUM(IF(author_id = owner_id, commits, NULL)) as commits_toself,
       COUNT(DISTINCT owner_id) as n_owners, COUNT(DISTINCT country_owner) as n_countries_o, 
       COUNT(DISTINCT IF(country_owner='ru' AND country_author != 'ru', owner_id, NULL)) as n_owners_ru, 
       COUNT(DISTINCT IF(country_owner='ua' AND country_author != 'ua', owner_id, NULL)) as n_owners_ua, 
       COUNT(DISTINCT IF(country_owner= country_author, owner_id, NULL)) as n_owners_home,
       CASE
         WHEN SUM(project_stars_cum) is NULL THEN 0
         ELSE SUM(project_stars_cum*commits)
       END commits_stars
FROM  `friends-foes.commits.puc` 

# merge project stars
LEFT JOIN 
    (SELECT DISTINCT repo_id, year+1 as year_1, stars,
                       SUM(stars) OVER(PARTITION BY repo_id ORDER BY DATE(year, 01, 01)) AS project_stars_cum
    FROM 
      (
      SELECT repo_id, 
            EXTRACT(YEAR FROM created_at) AS year, 
            COUNT(*) AS stars  
        FROM `ghtorrentmysql1906.MySQL1906.watchers` s
        LEFT JOIN 
          (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
        ON u.id = s.user_id 
        WHERE (country_current NOT IN ('ru', 'ua') OR country_current is null)
        GROUP BY repo_id,year
      )
    ) p_stars
ON p_stars.repo_id = parent_id AND p_stars.year_1 = year 
GROUP BY year, author_id
ORDER BY author_id, year;

CREATE TABLE `friends-foes.commits.ind_projects`
AS
# total and quality-weighted own projects 
SELECT p.year, p.owner_id, commits_received, commits_fromru, commits_fromua, commits_fromhome, commits_fromsef,
      n_contributors, n_countries_c, n_contributors_ru, n_contributors_ua, n_contributors_home, n_projects, 
      stars_own_projects
FROM 
(
SELECT year, owner_id, SUM(commits) as commits_received, 
       SUM(IF(country_author = 'ru' AND country_owner != 'ru', commits, NULL)) as commits_fromru, 
       SUM(IF(country_author = 'ua' AND country_owner != 'ua', commits, NULL)) as commits_fromua, 
       SUM(IF(country_owner = country_author, commits, NULL)) as commits_fromhome, 
       SUM(IF(owner_id = author_id, commits, NULL)) as commits_fromself, 
       COUNT(DISTINCT author_id) as n_contributors, COUNT(DISTINCT country_author) as n_countries_c, 
       COUNT(DISTINCT IF(country_author='ru' AND country_owner != 'ru', author_id, NULL)) as n_contributors_ru, 
       COUNT(DISTINCT IF(country_author='ua' AND country_owner != 'ua', author_id, NULL)) as n_contributors_ua, 
       COUNT(DISTINCT IF(country_owner= country_author, owner_id, NULL)) as n_contributors_home,
       COUNT(DISTINCT parent_id) as n_projects
       FROM `friends-foes.commits.puc` 
       GROUP BY owner_id, year
) p
LEFT JOIN 
(SELECT owner_id, year, SUM(stars) as stars_own_projects
FROM 
(
SELECT DISTINCT owner_id, parent_id, p.year, stars
FROM `friends-foes.commits.puc` p
LEFT JOIN 
    (SELECT repo_id, EXTRACT(YEAR FROM created_at) AS year, COUNT(*) AS stars  
      FROM `ghtorrentmysql1906.MySQL1906.watchers` s2
      LEFT JOIN 
      (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
       ON u.id = s2.user_id 
       WHERE country_current NOT IN ('ru', 'ua') OR country_current IS NULL
       GROUP BY repo_id,year
     ) own_stars
    ON p.parent_id = own_stars.repo_id AND p.year = own_stars.year
)
GROUP BY owner_id, year) s
ON s.owner_id = p.owner_id AND s.year = p.year 

'''
client.query(query, job_config=run) 

In [ ]:
# pre-2014 characteristics to use for CEM
query = '''
CREATE TABLE `friends-foes.quality.user_quality` 
AS
SELECT a.user_id, commits_pre, owners_pre, commits_toru_pre, commits_toua_pre, commits_tohome_pre, 
a.projects_pre, a.contributions_pre, a.contributors_pre, 
a.contributions_fromua_pre, a.contributions_fromru_pre, a.contributions_fromhome_pre, 
s.user_stars_cum, f.user_follow_cum
FROM
(
# previous commits and previous commits from home, ru, ua
# previous projects and contributors 
SELECT CASE WHEN c.user_id is not null THEN c.user_id ELSE p.user_id END user_id,
       commits_pre, owners_pre, commits_toru_pre, commits_tohome_pre, commits_toua_pre,
       projects_pre, contributors_pre, contributions_pre, 
       contributions_fromru_pre, contributions_fromua_pre, contributions_fromhome_pre
FROM 
  (SELECT user_id,
       MAX(commits_prev) as commits_pre, MAX(owners_prev) as owners_pre, 
       MAX(IF(country_partner = 'ru' AND country_user != 'ru', commits_prev_country, NULL)) as commits_toru_pre, 
       MAX(IF(country_partner = 'ua' AND country_user != 'ua', commits_prev_country, NULL)) as commits_toua_pre,
       MAX(IF(country_partner = country_user, commits_prev_country, NULL)) as commits_tohome_pre
       FROM `friends-foes.quality.user_commits` WHERE year < 2014 
       GROUP BY user_id) c
   FULL JOIN 
   (SELECT user_id, 
       MAX(projects_prev) as projects_pre, MAX(contributors_prev) as contributors_pre, 
       MAX(contributions_prev) as contributions_pre, 
       MAX(IF(country_partner = 'ru' AND country_user != 'ru', contributions_prev_country, NULL)) as contributions_fromru_pre, 
       MAX(IF(country_partner = 'ua' AND country_user != 'ua', contributions_prev_country, NULL)) as contributions_fromua_pre,
       MAX(IF(country_partner = country_user, contributions_prev_country, NULL)) as contributions_fromhome_pre
       FROM `friends-foes.quality.user_projects`  WHERE year < 2014
       GROUP BY user_id) p
    ON c.user_id = p.user_id 
) a
# stars  pre 2014
LEFT JOIN (SELECT user_id, MAX(user_stars_cum) As user_stars_cum
          FROM `friends-foes.quality.users_stars` WHERE year < 2014
          GROUP BY user_id) s
ON a.user_id = s.user_id 
# followers pre 2014
LEFT JOIN (SELECT user_id, MAX(user_follow_cum) As user_follow_cum
          FROM `friends-foes.quality.users_followers` WHERE year < 2014
          GROUP BY user_id) f
ON a.user_id = f.user_id 
'''
client.query(query, job_config=run) 

In [ ]:
# balance dataset with user_ids: those who existed before 2014; LEFT JOIN outcomes and pre-2014 characteristics
query = '''
DROP TABLE IF EXISTS `friends-foes.commits.ind_effects`;
CREATE TABLE `friends-foes.commits.ind_effects`
AS
SELECT a.user_id, a.year, year_created, month_created, cc2013_user, 
        district_en, dlk, ru_identity, ethnicities, mover, stayer, 
        commits_pre, owners_pre, commits_toru_pre, commits_toua_pre, commits_tohome_pre, projects_pre, 
        contributors_pre, contributions_pre, contributions_fromua_pre, contributions_fromru_pre, 
        contributions_fromhome_pre, user_stars_cum, user_follow_cum,
        commits_received,  commits_fromru, commits_fromua, commits_fromhome, commits_fromself,
        n_contributors, n_countries_c,  n_projects, stars_own_projects, 
        commits_sent, commits_stars, commits_toru, commits_toua, commits_tohome, commits_toself, 
        n_owners, n_owners_ru, n_owners_ua, 
        n_owners_home, n_countries_o
FROM
  (
  SELECT user_id, year, year_created, month_created, cc2013_user
  FROM 
  (SELECT DISTINCT id as user_id, year_created, month_created,
                  CASE WHEN cc2013 IN ('ua', 'ru') THEN cc2013 ELSE country_current END cc2013_user 
                  FROM `friends-foes.user_panel.sample_locations_panel`
                  WHERE year_created < 2014 AND year = 2013
  ) 
  CROSS JOIN 
  (SELECT * FROM UNNEST([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]) AS year) b
  WHERE year>= year_created
  ) a
# add additional information for ua users: location, identity, and movers/stayers
LEFT JOIN `friends-foes.user_panel.ua_users` b
ON a.user_id = b.id AND a.year = b.year
LEFT JOIN `friends-foes.user_panel.name_class_web` c
ON a.user_id = c.id 
LEFT JOIN `friends-foes.commits.movers` d
ON a.user_id = d.author_id
# add pre-2014 characteristics 
LEFT JOIN `friends-foes.quality.user_quality`e
ON a.user_id = e.user_id
# add outcomes 
LEFT JOIN `friends-foes.commits.ind_commits` f
ON a.user_id = f.author_id and a.year = f.year
LEFT JOIN `friends-foes.commits.ind_projects` g
ON a.user_id = g.owner_id and a.year = g.year
'''
client.query(query, job_config=run) 

In [ ]:
# select user_ids of directly treated (commits_toru(ua)_pre or contributors_from_ru(ua)_pre > 0) and indirectly treated (coathors/followers of those who were treated)
query = '''
DROP VIEW IF EXISTS `friends-foes.commits.treated_users`;

CREATE VIEW `friends-foes.commits.treated_users`
AS 
SELECT distinct user_id, cc2013_user from `friends-foes.commits.ind_effects`
where year = 2013 and 
(cc2013_user = 'ua' and (commits_toru_pre > 0 or contributions_fromru_pre > 0)
or 
cc2013_user = 'ru' and (commits_toua_pre > 0 or contributions_fromua_pre > 0)
)
'''
client.query(query, job_config=run) 

# select indirectly treated users
query = '''
SELECT distinct follower_id as id from `ghtorrentmysql1906.MySQL1906.followers` 
where EXTRACT(YEAR FROM created_at) < 2014 and user_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ua')
UNION distinct 
(SELECT distinct author_id as id from `friends-foes.commits.puc` where author_id <> owner_id and year < 2014 and owner_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ua'))
UNION distinct 
(SELECT distinct owner_id as id from `friends-foes.commits.puc` where author_id <> owner_id and year < 2014 and author_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ua'))
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'indirect_treated_ua.csv', encoding = 'utf-8', index=False)

query = '''
SELECT distinct follower_id as id from `ghtorrentmysql1906.MySQL1906.followers` 
where EXTRACT(YEAR FROM created_at) < 2014 and user_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ru')
UNION distinct 
(SELECT distinct author_id as id from `friends-foes.commits.puc` where author_id <> owner_id and year < 2014 and owner_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ru'))
UNION distinct 
(SELECT distinct owner_id as id from `friends-foes.commits.puc` where author_id <> owner_id and year < 2014 and author_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ru'))
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'indirect_treated_ru.csv', encoding = 'utf-8', index=False)

In [ ]:
query = '''
SELECT  distinct author_id as id from `friends-foes.commits.puc` where author_id <> owner_id and year < 2014 and owner_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ua')
UNION distinct 
(SELECT distinct owner_id as id from `friends-foes.commits.puc` where  author_id <> owner_id and year < 2014 and author_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ua'))
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv('database/stata_tmp/indirect_treated_ua_nf.csv', encoding = 'utf-8', index=False)

query = '''
SELECT distinct author_id as id from `friends-foes.commits.puc` where  author_id <> owner_id and year < 2014 and owner_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ru')
UNION distinct 
(SELECT distinct owner_id as id from `friends-foes.commits.puc` where  author_id <> owner_id and year < 2014 and author_id in (SELECT distinct user_id FROM `friends-foes.commits.treated_users` where cc2013_user = 'ru'))
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'indirect_treated_ru_nf.csv', encoding = 'utf-8', index=False)

In [ ]:
# all users active users as of 2013
query = '''
SELECT * 
FROM `friends-foes.commits.ind_effects` 
WHERE commits_pre > 0 
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_ind_effects_all.csv', encoding = 'utf-8', index=False)

In [ ]:
# not used anymore
# shorten the sample size 
query = '''
SELECT * 
FROM `friends-foes.commits.ind_effects` 
WHERE commits_pre > 0 AND (cc2013_user IN ('ua', 'ru') or commits_toru_pre > 0 or commits_toua_pre > 0)
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_ind_effects.csv', encoding = 'utf-8', index=False)
